## 0. 설치 (빈 venv 기준 · 한 번만)

macOS arm64용 PyTorch 기본 휠에 MPS 지원이 들어 있어 별도 인덱스 URL이 필요 없다.
설치 후 **커널을 재시작**하고 위에서부터 Run All.


In [1]:
# macOS(Apple Silicon): 기본 PyPI 휠에 MPS 백엔드 포함 — CUDA 인덱스 URL 불필요
%pip install -q torch

%pip install -q transformers peft trl datasets accelerate
%pip install -q langchain-chroma langchain-huggingface sentence-transformers chromadb


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. 환경·공통 규약

In [2]:
import os, json, re, random, gc, platform

# MPS에 아직 없는 연산은 CPU로 자동 폴백 (torch import 전에 설정해야 적용됨)
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# macOS: torch와 chromadb(onnxruntime)가 각자 libomp를 올려 교착/크래시 나는 문제 방지
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
# Chroma 텔레메트리 전송 끔(네트워크 대기로 멈추는 경우 방지)
os.environ["ANONYMIZED_TELEMETRY"] = "False"

# transformers가 torch와 안 맞는 torchvision을 불러오다 죽는 경우 방지(있어도 무해)
try:
    import transformers.utils.import_utils as _iu
    _iu._torchvision_available = False
    _iu.is_torchvision_available = lambda *a, **k: False
except Exception:
    pass

import torch

print("python arch:", platform.machine(), "| torch:", torch.__version__)
if platform.machine() != "arm64":
    print("⚠️ arm64가 아닌 파이썬입니다(Rosetta?). MPS를 쓰려면 arm64 파이썬으로 venv를 다시 만드세요.")

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

# 학습: MPS에서 fp16 학습은 NaN/불안정이 잦아 float32 사용 (1.5B + LoRA면 통합 메모리로 충분)
# 추론: MPS에서 float16이 빠르고 안정적
TRAIN_DTYPE = torch.float32
INFER_DTYPE = torch.float16 if DEVICE in ("mps", "cuda") else torch.float32

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "mps":
    torch.mps.manual_seed(SEED)
elif DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

def free_memory():
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()

print("device:", DEVICE, "| train dtype:", TRAIN_DTYPE, "| infer dtype:", INFER_DTYPE)

MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "qwen2_5_1_5b_koalpaca_lora_mps_sft"      # 학습 결과 저장 폴더
# 어댑터 후보: 이번에 학습한 것, 또는 이미 갖고 있는 것(_6에서 만든 것, Windows판 결과)
ADAPTER_CANDS = [
    os.path.join(OUTPUT_DIR, "adapter"),
    os.path.join("qwen2_5_1_5b_koalpaca_lora_mps_sft_1", "adapter"),
    os.path.join("qwen2_5_1_5b_koalpaca_lora_sft", "adapter"),
]

# 학습·추론이 함께 쓰는 단일 시스템 프롬프트
SYSTEM_PROMPT = (
    "당신은 한국어 법률 상담 어시스턴트입니다. 반드시 한국어로만 답하세요. "
    "제공된 [문맥]만을 근거로 정확하게 답하세요."
)

python arch: arm64 | torch: 2.14.0
device: mps | train dtype: torch.float32 | infer dtype: torch.float16


## 2. 모델 로드

MPS에서는 `device_map="auto"`가 CPU/디스크로 분산 배치하는 경우가 있어, 로드 후 `.to(DEVICE)`로 명시적으로 올린다.


In [4]:
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM

ADAPTER_PATH = find_adapter()
assert ADAPTER_PATH, "어댑터를 찾지 못했습니다. §2가 정상 실행됐는지 확인하세요."
print("어댑터:", ADAPTER_PATH)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
model = AutoPeftModelForCausalLM.from_pretrained(ADAPTER_PATH, dtype=INFER_DTYPE)
model = model.merge_and_unload()     # LoRA를 베이스에 병합 → 추론 속도 향상
model.to(DEVICE)
model.eval()
print("모델 로드 완료 · device:", next(model.parameters()).device)

어댑터: qwen2_5_1_5b_koalpaca_lora_mps_sft/adapter


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

모델 로드 완료 · device: mps:0


## 3. RAG 검색 — 로컬 임베딩 (API 키 불필요)

같은 폴더의 `legalQA.jsonlines`(한 줄 = `{"title","question","answer",...}`)로 벡터DB를 만든다.
한 번 만들면 `legal_db_local/`에 저장돼 다음 실행부터는 바로 로드된다.

- 단계마다 시각을 찍어 **어디서 멈추는지 바로 보이게** 했다.
- 임베딩 모델(110M)은 작아서 CPU로도 2천 건이 1분 안팎이다. 멈춤 원인을 줄이려고 기본은 CPU, 원한다면 `EMB_DEVICE = DEVICE`.
- 중간에 끊겨 반쯤 만들어진 DB는 자동으로 지우고 다시 만든다.


In [5]:
import time, shutil

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

log("1) 라이브러리 import")
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
log("   langchain_huggingface OK")
from langchain_chroma import Chroma
log("   langchain_chroma OK")

EMB_ID     = "jhgan/ko-sroberta-multitask"
EMB_DEVICE = "cpu"          # 문제 없으면 DEVICE("mps")로 바꿔도 됨
LEGAL_QA_PATH = next((p for p in ["legalQA.jsonlines", "legalqa.jsonlines", "legalQA.txt"]
                      if os.path.exists(p)), "legalQA.jsonlines")
DB_PATH = "legal_db_local"
TOP_K = 3

log(f"2) 임베딩 모델 다운로드 확인 ({EMB_ID}, 약 440MB — 처음 한 번만)")
from huggingface_hub import snapshot_download
snapshot_download(EMB_ID)
log("   다운로드/캐시 OK")

log(f"3) 임베딩 모델 로드 (device={EMB_DEVICE})")
embeddings = HuggingFaceEmbeddings(
    model_name=EMB_ID,
    model_kwargs={"device": EMB_DEVICE},
    encode_kwargs={"batch_size": 64},
)
t = time.time(); embeddings.embed_query("테스트 문장입니다")
log(f"   테스트 임베딩 {time.time()-t:.2f}s")

def load_qa(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            o = json.loads(line)
            q = (o.get("question") or "").strip()
            a = (o.get("answer") or "").strip()
            if q and a:
                rows.append({"q": q, "a": a, "title": (o.get("title") or "").strip()})
    return rows

log("4) 벡터DB 준비")
vectorstore = None
if os.path.isdir(DB_PATH) and os.listdir(DB_PATH):
    vs = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
    n = vs._collection.count()
    if n > 0:
        vectorstore = vs
        log(f"   기존 벡터DB 로드: {DB_PATH} (문서 {n}건)")
    else:
        log("   비어 있거나 깨진 DB → 삭제 후 재생성")
        del vs; shutil.rmtree(DB_PATH)

if vectorstore is None:
    assert os.path.exists(LEGAL_QA_PATH), f"'{LEGAL_QA_PATH}' 를 이 노트북과 같은 폴더에 두세요."
    qa = load_qa(LEGAL_QA_PATH)
    docs = [Document(page_content=f"[질문] {r['q']}\n[답변] {r['a']}",
                     metadata={"title": r["title"]}) for r in qa]
    log(f"   문서 {len(docs)}건 임베딩 시작")
    vectorstore = Chroma(persist_directory=DB_PATH, embedding_function=embeddings)
    B = 256
    for i in range(0, len(docs), B):
        vectorstore.add_documents(docs[i:i+B])
        log(f"   {min(i+B, len(docs))}/{len(docs)}")
    log(f"   생성 완료: {DB_PATH}")

def retrieve(question, k=TOP_K):
    return [d.page_content for d in vectorstore.similarity_search(question, k=k)]

log("5) 검색 테스트")
for c in retrieve("전세 보증금을 집주인이 안 돌려줘요. 어떻게 하나요?"):
    print(" -", c[:90].strip(), "...")


[11:40:40] 1) 라이브러리 import
[11:40:40]    langchain_huggingface OK
[11:40:41]    langchain_chroma OK
[11:40:41] 2) 임베딩 모델 다운로드 확인 (jhgan/ko-sroberta-multitask, 약 440MB — 처음 한 번만)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 21 files:   0%|          | 0/21 [00:00<?, ?it/s]

[11:44:22]    다운로드/캐시 OK
[11:44:22] 3) 임베딩 모델 로드 (device=cpu)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[11:44:28]    테스트 임베딩 0.50s
[11:44:28] 4) 벡터DB 준비
[11:44:28]    문서 1830건 임베딩 시작
[11:44:30]    256/1830
[11:44:33]    512/1830
[11:44:36]    768/1830
[11:44:39]    1024/1830
[11:44:41]    1280/1830
[11:44:44]    1536/1830
[11:44:47]    1792/1830
[11:44:48]    1830/1830
[11:44:48]    생성 완료: legal_db_local
[11:44:48] 5) 검색 테스트
 - [질문] 보증금을 못 받아 살림살이 일부를 놔두고 이사를 하였습니다. 그런데 집주인이 월세를 내놓으라고 하네요. 너무 억울합니다. 어떻게 해야 하나요?
[답변] ...
 - [질문] 저는 주택임차인인데 집주인으로부터 임대기간이 끝나기 불과 며칠 전에 임대보증금을 인상하여 주거나 그것이 불가능하면 기간 만료와 동시에 집을 비워 달라는 ...
 - [질문] 저는 그 집에 살고 있던 사람을 소유자로 알고 임대차계약을 체결하고 입주하여 주민등록을 마쳤습니다. 그런데 나중에 소유자는 따로 있고 저랑 계약을 체결한 ...


## 4. RAG 추론 — 검색 근거를 붙여 답하기

In [6]:
def chat_with_model_rag(question, k=TOP_K, max_new_tokens=512):
    docs = retrieve(question, k)
    context = "\n\n".join(docs)
    print(f"\n[질문] {question}\n[검색된 문서]")
    for i, d in enumerate(docs, 1):
        print(f"  {i}. {d[:90].strip()} ...")

    user = f"다음 법률 문맥을 참고하여 답변하세요:\n\n[법률 문맥]\n{context}\n\n[질문]\n{question}"
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        gen = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False,                 # 재현 가능하도록 greedy
                             repetition_penalty=1.1,          # 소형 모델 반복 억제
                             pad_token_id=tokenizer.eos_token_id)
    ans = tokenizer.decode(gen[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("[답변]\n" + ans.strip())
    if DEVICE == "mps":
        torch.mps.empty_cache()
    return ans.strip()

for q in [
    "신호위반 차량이 저를 들이받고 도주했습니다. 가해자가 무보험이라는데 어떻게 되나요?",
    "회사에서 갑자기 해고 통보를 받았는데 해고예고수당을 받을 수 있나요?",
]:
    chat_with_model_rag(q)


[질문] 신호위반 차량이 저를 들이받고 도주했습니다. 가해자가 무보험이라는데 어떻게 되나요?
[검색된 문서]
  1. [질문] 저는 丙이 운전하던 차량이 신호를 위반하여 횡단보도상을 횡단하던 저를 충격 하는 사고로 인하여 장애가능성이 있는 상해를 입었습니다. 그런데 丙은 친구인 ...
  2. [질문] 저는 약간의 술을 마시고 도로를 주행하던 중 무단횡단 하던 피해자를 발견하지 못하여 중상을 입히는 사고를 냈습니다. 그런데 저는 일단 그 자리를 피한 후 ...
  3. [질문] 저는 甲보험회사와 개인용자동차종합보험계약을 체결하면서 무보험자동차에 의한 상해특약을 체결하였습니다. 그 후 저는 피보험차량을 운전하다가 빗길에 미끄러지면 ...
[답변]
저는 신호위반 차량이 저를 들이받고 도주했습니다. 가해자가 무보험이며, 가해자가 무면허 운전을 했다고 합니다. 만약 가해자가 무보험이라면, 가해자가 무면허 운전을 했다면, 가해자가 무보험자동차에 의한 상해특약을 체결해야 합니다. 그러나 가해자가 무면허 운전을 했다면, 가해자가 무보험자동차에 의한 상해특약을 체결할 수 없습니다. 따라서 가해자가 무보험자동차에 의한 상해특약을 체결할 수 없습니다.

[질문] 회사에서 갑자기 해고 통보를 받았는데 해고예고수당을 받을 수 있나요?
[검색된 문서]
  1. [질문] 저는 얼마 전 사업주로부터 아무런 이유 없이 일방적인 해고통지를 받았습니다. 주위에서는 해고를 하더라도 일정한 절차에 의하여야 하며 해고예고수당 또한 지 ...
  2. [질문] 의료업을 경영하는 X회사로부터 월 130만원을 지급받기로 하고 고용되어 사무직원으로 일하던 중 5개월쯤 되어 X회사로부터 일방적으로 해고당하였습니다. 상 ...
  3. [질문] 저는 X회사에 2005년경에 입사하여 4년을 근무하다가 2009년 말경 회사측의 직권면직으로 해고되었는데, 당시 아무런 이의 없이 퇴직금을 수령하였습니다 ...
[답변]
(1) 근로기준법 제26조에 의하여, 사용자는 근로자를 해고하고자 할 때에는 적어도 30일 전에